### Import

In [1]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 30
LEVEL = "high"
SEED = 1

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

BASE_PATH = "/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result"


✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=30
✅ 시뮬레이션 초기화 완료: S=30, Randomness='high', Random Seed=1, M1=722.00, M2=1957.00


In [2]:
print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, OBJ_IND = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)
print("-"*100)

print("[Holistic Aggregation Model optimization]")
x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, OBJ_HOL = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)
print("-"*100)

[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter Username
Set parameter LicenseID to value 2681721
Academic license - for non-commercial use only - expires 2026-06-24
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  20%|██        | 1/5 [00:00<00:00,  4.31it/s]

Optimal solution found for target_i=0! Objective value: 222190.54193014003
Set parameter MIPGap to value 1e-07


...

Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00,  4.42it/s]

Optimal solution found for target_i=3! Objective value: 454586.0716013976
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:01<00:00,  4.20it/s]

Optimal solution found for target_i=4! Objective value: 170336.98758043215
----------------------------------------------------------------------------------------------------
[Holistic Aggregation Model optimization]
Set parameter MIPGap to value 0.0001


Optimal solution found! Objective value: 1730223.4906900022
----------------------------------------------------------------------------------------------------


In [3]:
# internal pool price bounds

# Risk Neutral
MIN_BOUND = {t: np.mean([P_RT[t, s] for s in range(S)]) for t in range(T)}
MAX_BOUND = {t: np.mean([P_PN[t, s] for s in range(S)]) for t in range(T)}

# # Risk Averse
# MIN_BOUND = {t: np.max([P_RT[t, s] for s in range(S)]) for t in range(T)}
# MAX_BOUND = {t: np.min([P_PN[t, s] for s in range(S)]) for t in range(T)}
# for t in range(T):
#     if MIN_BOUND[t] >= MAX_BOUND[t]:
#         avg_value = (MIN_BOUND[t] + MAX_BOUND[t]) / 2
#         MIN_BOUND[t] = avg_value
#         MAX_BOUND[t] = avg_value

# Risk Seeking
# MIN_BOUND = {t: np.min([P_RT[t, s] for s in range(S)]) for t in range(T)}
# MAX_BOUND = {t: np.max([P_PN[t, s] for s in range(S)]) for t in range(T)}

### Endogeneous

In [34]:
def optimize_endogenous(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2):
    
    model = gp.Model("DER_Aggregation_Endogenous")
    model.setParam("Heuristics", 0.3)
    model.setParam("TimeLimit", 60*30)
    
    # Decision Variables
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ep") 
    ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="em")  
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")  
    dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")  
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") 
    zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1")
    phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2")
    phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")
    phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5")
    phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")
    phi8 = model.addVars(T, S, vtype=GRB.BINARY, name="phi8")
    rhop = model.addVars(T, vtype=GRB.CONTINUOUS, name="rhop")
    rhom = model.addVars(T, vtype=GRB.CONTINUOUS, name="rhom")
    
    model.update()
    
    # Objective Function
    obj = (
        # Day-ahead revenue
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
        # Expected real-time revenue and penalty costs
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - 
            P_PN[t, s] * ym[i, t, s] +
            rhop[t] * dp[i, t, s] -
            rhom[t] * dm[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S))
    )
    
    model.setObjective(obj, GRB.MAXIMIZE)
    
    # Constraints
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        model.addConstr(zd[i, t, s] <= z[i, t, s])
        model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
        model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + zc[i, t, s] - zd[i, t, s])
        
        # Logical constraints
        model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi2[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi3[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi4[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi5[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi6[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi7[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi7[i, t, s]))

    for i, s in product(range(I), range(S)):
        model.addConstr(z[i, 0, s] == K0[i])

    for t, s in product(range(T), range(S)):
        model.addConstr(gp.quicksum(dp[i, t, s] for i in range(I)) == gp.quicksum(dm[i, t, s] for i in range(I)))
        model.addConstr(gp.quicksum(yp[i, t, s] for i in range(I)) <= M2 * phi8[t, s])
        model.addConstr(gp.quicksum(ym[i, t, s] for i in range(I)) <= M2 * (1 - phi8[t, s]))
    
    for t in range(T):
        model.addConstr(rhop[t] == rhom[t])
        # model.addConstr(rhop[t] <= 600) # 물리적인 제약
        model.addConstr(rhop[t] >= MIN_BOUND[t]) # internal pool에서 파는 가격은 RT보다 커야함
        # model.addConstr(rhom[t] >= P_DA[t])
        model.addConstr(rhom[t] <= MAX_BOUND[t]) # internal pool에서 사는 가격은 PN보다 작아야함
    
    # model.addConstr(
    #     gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    #     gp.quicksum((1/S) * (
    #         P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] 
    #         + rhop[t] * dp[i, t, s] - rhom[t] * dm[i, t, s]
    #     ) for i in range(I) for t in range(T) for s in range(S)) <= OBJ_HOL
    # )

    # for i in range(I):
    #     model.addConstr(
    #         gp.quicksum(P_DA[t] * x[i, t] for t in range(T)) +
    #         gp.quicksum((1/S) * (
    #             P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] +
    #             rhop[t] * dp[i, t, s] - rhom[t] * dm[i, t, s]
    #         ) for t in range(T) for s in range(S)) >= OBJ_IND[i]
    #     )

    # Optimize
    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {model.objVal}")
    else:
        print("No optimal solution found.")
    
    # Extract solution
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    rhop_sol = np.array([rhop[t].X for t in range(T)])
    rhom_sol = np.array([rhom[t].X for t in range(T)])

    # Aggregate values
    a_sol = np.sum(x_sol, axis=0)  # Aggregate day-ahead commitment
    bp_sol = np.sum(yp_sol, axis=0)  # Aggregate real-time selling
    bm_sol = np.sum(ym_sol, axis=0)  # Aggregate real-time penalty
    dp_agg = np.sum(dp_sol, axis=0)  # Aggregate internal pool selling
    dm_agg = np.sum(dm_sol, axis=0)  # Aggregate internal pool buying
    
    return (x_sol, a_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, 
            dp_sol, dm_sol, bp_sol, bm_sol, dp_agg, dm_agg, model.objVal, rhop_sol, rhom_sol)

In [ ]:
x, a, yp, ym, z, zc, zd, dp, dm, bp, bm, dp_agg, dm_agg, obj_val, rhop, rhom = optimize_endogenous(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)

Set parameter Heuristics to value 0.3
Set parameter TimeLimit to value 1800
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  1800
Heuristics  0.3

Optimize a model with 74382 rows, 51438 columns and 188886 nonzeros
Model fingerprint: 0xcd158c6d
Model has 7200 quadratic objective terms
Variable types: 25518 continuous, 25920 integer (25920 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [2e+00, 2e+02]
  QObjective range [7e-02, 7e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+03]
Presolve removed 28568 rows and 17056 columns
Presolve time: 0.34s
Presolved: 56621 rows, 39787 columns, 154445 nonzeros
Presolved model has 5403 bilinear constraint(s)

Solving non-convex MIQCP

Variable types: 22785 continuou

In [ ]:
# # 한 시나리오에 대해서 한명의 해 
# i = 3
# scen = 11

# header = (
#     f"{'s':>2} {'t':>2} | "
#     f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
#     f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
#     + "-" * 90
# )
# print(header)

# for s, t in product(range(scen,scen+1), range(8,22)):
#     # print(header)
#     # individual
#     # print(
#     #     f"{s:>2} {t:>2} | "
#     #     f"{R[i, t, s]:>8.2f} {x_ind[i][t]:>8.2f} {yp_ind[i][t, s]:>8.2f} {ym_ind[i][t, s]:>8.2f} "
#     #     f"{0:>8.2f} {0:>8.2f} {zc_ind[i][t, s]:>8.2f} {zd_ind[i][t, s]:>8.2f} {z_ind[i][t, s]:>8.2f}"
#     #     )
#     # endogenous
#     print(
#         f"{s:>2} {t:>2} | "
#         f"{R[i, t, s]:>8.2f} {x[i, t]:>8.2f} {yp[i, t, s]:>8.2f} {ym[i, t, s]:>8.2f} "
#         f"{dp[i, t, s]:>8.2f} {dm[i, t, s]:>8.2f} {zc[i, t, s]:>8.2f} {zd[i, t, s]:>8.2f} {z[i, t, s]:>8.2f}"
#     )
#     # holistic
#     # print(
#     #     f"{s:>2} {t:>2} | "
#     #     f"{R[i, t, s]:>8.2f} {x_hol[i, t]:>8.2f} {ep_hol[i, t, s]:>8.2f} {em_hol[i, t, s]:>8.2f} "
#     #     f"{dp_hol[i, t, s]:>8.2f} {dm_hol[i, t, s]:>8.2f} {zc_hol[i, t, s]:>8.2f} {zd_hol[i, t, s]:>8.2f} {z_hol[i, t, s]:>8.2f}"
#     #     )
#     # print()

In [ ]:
# # 한 시나리오에 대해서 전체합 
# scen = 0

# header = (
#     f"{'s':>2} {'t':>2} | "
#     f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
#     f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
#     + "-" * 90
# )
# print(header)

# for s, t in product(range(scen, scen+1), range(8, 22)):
#     print(
#         f"{s:>2} {t:>2} | "
#         f"{R[:, t, s].sum():>8.2f} {x[:, t].sum():>8.2f} {yp[:, t, s].sum():>8.2f} {ym[:, t, s].sum():>8.2f} "
#         f"{dp[:, t, s].sum():>8.2f} {dm[:, t, s].sum():>8.2f} {zc[:, t, s].sum():>8.2f} {zd[:, t, s].sum():>8.2f} {z[:, t, s].sum():>8.2f}"
#     )
#     # print(
#     #         f"{s:>2} {t:>2} | "
#     #         f"{R[:, t, s].sum():>8.2f} {x_hol[:, t].sum():>8.2f} {yp_hol[:, t, s].sum():>8.2f} {ym_hol[:, t, s].sum():>8.2f} "
#     #         f"{dp_hol[:, t, s].sum():>8.2f} {dm_hol[:, t, s].sum():>8.2f} {zc_hol[:, t, s].sum():>8.2f} {zd_hol[:, t, s].sum():>8.2f} {z_hol[:, t, s].sum():>8.2f}"
#     # )

In [ ]:
# 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x[:, t].sum()
    yp_avg = np.mean([yp[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    46.37     0.00    14.90     0.00     0.00     0.00    35.17     3.70    15.67
 9 |   169.57    24.00    12.47     0.00     0.17     0.17   133.23     0.13    47.13
10 |   438.10   233.00   101.73     0.00    20.00    20.00   156.77    53.40   180.23
11 |   661.03   145.00   486.97     0.00     4.53     4.53   106.73    77.67   283.60
12 |   875.37   256.00   615.87     0.00    29.00    29.00    81.90    78.40   312.67
13 |  1259.70   591.00   716.83     0.00    21.67    21.67   112.60   160.73   316.17
14 |  1365.03  1203.00   117.90    59.40   531.93   531.93   159.80    56.27   268.03
15 |   868.27   232.00   722.00     0.00     0.00     0.00    35.70   121.43   371.57
16 |   800.70   679.00   219.20     0.00    88.47    88.47    60.00   157.50   285.83
17 |   763.67   260.00   614.13     0.00     0.47

In [ ]:
# 만약 hol 변수들도 출력하고 싶다면 아래 주석 해제
print("\n=== HOL Variables (평균) ===")
header_hol = (
    f"{'t':>2} | "
    f"{'R':>8} {'x_hol':>8} {'y+_hol':>8} {'y-_hol':>8} "
    f"{'d+_hol':>8} {'d-_hol':>8} {'zc_hol':>8} {'zd_hol':>8} {'z_hol':>8}\n"
    + "-" * 95
)
print(header_hol)

for t in range(8, 22):
    # HOL 변수들의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_hol_sum = x_hol[:, t].sum()  # x_hol은 시나리오와 무관
    ep_hol_avg = np.mean([ep_hol[:, t, s].sum() for s in range(S)])
    em_hol_avg = np.mean([em_hol[:, t, s].sum() for s in range(S)])
    dp_hol_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_hol_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_hol_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_hol_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_hol_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_hol_sum:>8.2f} {ep_hol_avg:>8.2f} {em_hol_avg:>8.2f} "
        f"{dp_hol_avg:>8.2f} {dm_hol_avg:>8.2f} {zc_hol_avg:>8.2f} {zd_hol_avg:>8.2f} {z_hol_avg:>8.2f}"
    )


=== HOL Variables (평균) ===
 t |        R    x_hol   y+_hol   y-_hol   d+_hol   d-_hol   zc_hol   zd_hol    z_hol
-----------------------------------------------------------------------------------------------
 8 |    46.37     0.00    13.13     0.00     0.00     0.00    37.37     4.13    15.67
 9 |   169.57    23.00    11.87     0.00     0.13     0.13   135.37     0.67    48.90
10 |   438.10   206.00    83.40     0.00    23.43    23.43   176.33    27.63   183.60
11 |   661.03     0.00   590.37     0.00     0.00     0.00   135.23    64.57   332.30
12 |   875.37     0.00   809.80     0.00     0.00     0.00    88.30    22.73   402.97
13 |  1259.70     0.00  1619.63     0.00     0.00     0.00     3.63   363.57   468.53
14 |  1365.03   830.00   217.63    50.10   537.73   537.73   367.50     0.00   108.60
15 |   868.27     0.00  1306.60     0.00     0.00     0.00     6.20   444.53   476.10
16 |   800.70   417.00   166.77     0.00   364.47   364.47   218.73     1.80    37.77
17 |   763.67   

In [ ]:
for t in range(T):
    dp_avg = np.mean([dp[:, t, s].sum() for s in range(S)]); dm_avg = np.mean([dm[:, t, s].sum() for s in range(S)])

    print(t, 'RT', round(MIN_BOUND[t], 3), 'rhop', round(rhop[t], 3), 'dp', round(dp_avg, 3))
    print(t, 'PN', round(MAX_BOUND[t], 3), 'rhom', round(rhom[t], 3), 'dm', round(dm_avg, 3))
    print()

0 RT 61.984 rhop 61.984 dp 0.0
0 PN 148.649 rhom 61.984 dm 0.0

1 RT 88.609 rhop 88.609 dp 0.0
1 PN 141.984 rhom 88.609 dm 0.0

2 RT 77.887 rhop 77.887 dp 0.0
2 PN 126.208 rhom 77.887 dm 0.0

3 RT 70.584 rhop 70.584 dp 0.0
3 PN 119.527 rhom 70.584 dm 0.0

4 RT 68.942 rhop 119.554 dp 0.0
4 PN 119.554 rhom 119.554 dm 0.0

5 RT 85.134 rhop 85.134 dp 0.0
5 PN 133.657 rhom 85.134 dm 0.0

6 RT 74.891 rhop 74.891 dp 0.0
6 PN 131.454 rhom 74.891 dm 0.0

7 RT 89.499 rhop 149.669 dp 0.0
7 PN 149.669 rhom 149.669 dm 0.0

8 RT 110.459 rhop 110.459 dp 0.0
8 PN 177.972 rhom 110.459 dm 0.0

9 RT 101.732 rhop 184.684 dp 0.167
9 PN 184.684 rhom 184.684 dm 0.167

10 RT 113.639 rhop 113.639 dp 20.0
10 PN 194.179 rhom 113.639 dm 20.0

11 RT 142.623 rhop 223.613 dp 4.533
11 PN 223.613 rhom 223.613 dm 4.533

12 RT 174.009 rhop 264.94 dp 29.0
12 PN 264.94 rhom 264.94 dm 29.0

13 RT 265.696 rhop 398.544 dp 21.667
13 PN 398.544 rhom 398.544 dm 21.667

14 RT 118.558 rhop 236.984 dp 531.933
14 PN 236.984 rhom 23

In [ ]:
profit = np.zeros(I)
for i in range(I):
    profit[i] = (
        # Day-ahead revenue
        sum(P_DA[t] * x[i, t] for t in range(T)) +
        # Expected real-time revenue and penalty costs
        sum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - 
            P_PN[t, s] * ym[i, t, s] +
            rhop[t] * dp[i, t, s] -
            rhom[t] * dm[i, t, s]
        ) for t in range(T) for s in range(S))
    )

sum_=0
for i in range(I):
    print('ind', i, round(OBJ_IND[i],3))
    sum_ += OBJ_IND[i]
    print('model', i, round(profit[i],3))
    print()

print('OBJ_IND', round(sum_, 3))
print('OBJ_MODEL', round(obj_val, 3))
print('OBJ_HOL', round(OBJ_HOL,3))
print("------------------------")
print('verify', round(profit[:].sum(),3))


ind 0 222190.542
model 0 173215.486

ind 1 339434.18
model 1 329946.116

ind 2 414349.006
model 2 381445.641

ind 3 454586.072
model 3 442266.933

ind 4 170336.988
model 4 181800.429

OBJ_IND 1600896.788
OBJ_MODEL 1508674.605
OBJ_HOL 1730223.491
------------------------
verify 1508674.605


In [ ]:
def compare_models(x, ep, em, dp, dm, rhop, rhom, obj_val,
                   x_hol, ep_hol, em_hol, dp_hol, dm_hol, OBJ_HOL,
                   P_DA, P_RT, P_PN, I, T, S):
    """
    Compare detailed profits between endogenous and holistic models
    """
    
    print("=" * 60)
    print("MODEL COMPARISON ANALYSIS")
    print("=" * 60)
    
    # 1. Overall objective value comparison
    print(f"\n1. OBJECTIVE VALUE COMPARISON")
    print(f"   Individual Model: {sum_:>12.2f}")
    print(f"   Endogenous Model: {obj_val:>12.2f}")
    print(f"   Holistic Model:   {OBJ_HOL:>12.2f}")
    
    # 2. Internal pool market clearing check
    print(f"\n2. INTERNAL POOL MARKET CLEARING CHECK")
    dp_sum = np.sum(dp, axis=0)  # (T, S) shape으로 만들기 - i에 대한 sum
    dm_sum = np.sum(dm, axis=0)  # (T, S) shape으로 만들기 - i에 대한 sum

    # Daily total check
    total_sell = np.sum(rhop.reshape(-1, 1) * dp_sum)  # (T, 1) * (T, S)
    total_buy = np.sum(rhom.reshape(-1, 1) * dm_sum)   # (T, 1) * (T, S)
    print(f"   Daily Total:")
    print(f"     Total Sell Value: {total_sell:>12.2f}")
    print(f"     Total Buy Value:  {total_buy:>12.2f}")
    print(f"     Net Transfer:     {total_sell - total_buy:>12.2f}")
    print(f"     Balanced: {'✓' if abs(total_sell - total_buy) < 1e-6 else '✗'}")

    # Hourly check (show violations if any)
    print(f"\n   Hourly Check:")
    violations = []
    for t in range(T):
        hourly_sell = np.sum(rhop[t] * dp_sum[t, :])  # 시간 t에서 모든 시나리오 합
        hourly_buy = np.sum(rhom[t] * dm_sum[t, :])   # 시간 t에서 모든 시나리오 합
        net_transfer = hourly_sell - hourly_buy
        
        if abs(net_transfer) > 1e-6:
            violations.append((t, hourly_sell, hourly_buy, net_transfer))

    if violations:
        print(f"     Found {len(violations)} hourly violations:")
        print(f"     {'Hour':>4} | {'Sell':>12} | {'Buy':>12} | {'Net':>12}")
        print(f"     {'-'*4} | {'-'*12} | {'-'*12} | {'-'*12}")
        for t, sell, buy, net in violations[:10]:  # Show first 10
            print(f"     {t:>4} | {sell:>12.2f} | {buy:>12.2f} | {net:>12.2f}")
        if len(violations) > 10:
            print(f"     ... and {len(violations) - 10} more")
    else:
        print(f"     All hourly balances satisfied ✓")

    # Market clearing quantity check (without prices)
    print(f"\n   Quantity Balance Check:")
    qty_violations = []
    for t in range(T):
        for s in range(S):
            sell_qty = dp_sum[t, s]
            buy_qty = dm_sum[t, s]
            if abs(sell_qty - buy_qty) > 1e-6:
                qty_violations.append((t, s, sell_qty, buy_qty))

    if qty_violations:
        print(f"     Found {len(qty_violations)} quantity violations:")
        print(f"     {'Hour':>4} | {'Scen':>4} | {'Sell Qty':>12} | {'Buy Qty':>12} | {'Diff':>12}")
        print(f"     {'-'*4} | {'-'*4} | {'-'*12} | {'-'*12} | {'-'*12}")
        for t, s, sell, buy in qty_violations[:10]:
            print(f"     {t:>4} | {s:>4} | {sell:>12.2f} | {buy:>12.2f} | {sell-buy:>12.2f}")
        if len(qty_violations) > 10:
            print(f"     ... and {len(qty_violations) - 10} more")
    else:
        print(f"     All quantity balances satisfied ✓")
        
    # 3. Day-ahead profit comparison
    print(f"\n3. DAY-AHEAD PROFIT COMPARISON")
    da_profit_endo = np.sum([P_DA[t] * np.sum(x[:, t]) for t in range(T)])
    da_profit_hol = np.sum([P_DA[t] * np.sum(x_hol[:, t]) for t in range(T)])
    print(f"   Endogenous Model: {da_profit_endo:>12.2f}")
    print(f"   Holistic Model:   {da_profit_hol:>12.2f}")
    print(f"   Difference:       {da_profit_endo - da_profit_hol:>12.2f}")
    
    # 4. Day-ahead commitment quantity comparison
    print(f"\n4. DAY-AHEAD COMMITMENT QUANTITY")
    da_qty_endo = np.sum(x)
    da_qty_hol = np.sum(x_hol)
    print(f"   Endogenous Model: {da_qty_endo:>12.2f}")
    print(f"   Holistic Model:   {da_qty_hol:>12.2f}")
    print(f"   Difference:       {da_qty_endo - da_qty_hol:>12.2f}")
    
    # 5. Real-time profit comparison
    print(f"\n5. REAL-TIME PROFIT COMPARISON")
    rt_profit_endo = np.sum([P_RT[t, s] * np.sum(ep[:, t, s]) for t in range(T) for s in range(S)]) / S
    rt_profit_hol = np.sum([P_RT[t, s] * np.sum(ep_hol[:, t, s]) for t in range(T) for s in range(S)]) / S
    print(f"   Endogenous Model: {rt_profit_endo:>12.2f}")
    print(f"   Holistic Model:   {rt_profit_hol:>12.2f}")
    print(f"   Difference:       {rt_profit_endo - rt_profit_hol:>12.2f}")
    
    # 6. Real-time quantity comparison
    print(f"\n6. REAL-TIME QUANTITY COMPARISON")
    rt_qty_endo = np.mean(np.sum(ep, axis=(0, 1)))
    rt_qty_hol = np.mean(np.sum(ep_hol, axis=(0, 1)))
    print(f"   Endogenous Model: {rt_qty_endo:>12.2f}")
    print(f"   Holistic Model:   {rt_qty_hol:>12.2f}")
    print(f"   Difference:       {rt_qty_endo - rt_qty_hol:>12.2f}")
    
    # 7. Penalty comparison
    print(f"\n7. PENALTY COMPARISON")
    penalty_endo = np.sum([P_PN[t, s] * np.sum(em[:, t, s]) for t in range(T) for s in range(S)]) / S
    penalty_hol = np.sum([P_PN[t, s] * np.sum(em_hol[:, t, s]) for t in range(T) for s in range(S)]) / S
    print(f"   Endogenous Model: {penalty_endo:>12.2f}")
    print(f"   Holistic Model:   {penalty_hol:>12.2f}")
    print(f"   Difference:       {penalty_endo - penalty_hol:>12.2f}")
    
    # 8. Internal pool trading comparison
    print(f"\n8. INTERNAL POOL TRADING")
    pool_trade_endo = np.mean(np.sum(dp + dm, axis=(0, 1)))
    pool_trade_hol = np.mean(np.sum(dp_hol + dm_hol, axis=(0, 1)))
    print(f"   Endogenous Model: {pool_trade_endo:>12.2f}")
    print(f"   Holistic Model:   {pool_trade_hol:>12.2f}")
    print(f"   Difference:       {pool_trade_endo - pool_trade_hol:>12.2f}")
    
    # 9. Individual DER comparison (first few DERs)
    print(f"\n9. INDIVIDUAL DER PROFIT COMPARISON")
    print(f"   {'DER':>3} | {'Individual':>12} | {'Endogenous':>12} | {'Holistic':>12} |")
    print(f"   {'-'*3} | {'-'*12} | {'-'*12} | {'-'*12} |")
    
    for i in range(I):
        profit_ind = OBJ_IND[i]
        profit_endo = (
            sum(P_DA[t] * x[i, t] for t in range(T)) +
            sum((1/S) * (P_RT[t, s] * ep[i, t, s] - P_PN[t, s] * em[i, t, s] +
            rhop[t] * dp[i, t, s] - rhom[t] * dm[i, t, s]) 
                for t in range(T) for s in range(S))
        )
        profit_hol = (
            sum(P_DA[t] * x_hol[i, t] for t in range(T)) +
            sum((1/S) * (P_RT[t, s] * ep_hol[i, t, s] - P_PN[t, s] * em_hol[i, t, s]) 
                for t in range(T) for s in range(S))
        )
        print(f"   {i:>3} | {profit_ind:>12.2f} | {profit_endo:>12.2f} | {profit_hol:>12.2f} |")
    
    print("=" * 60)

# 사용법
compare_models(x, yp, ym, dp, dm, rhop, rhom, obj_val,
               x_hol, ep_hol, em_hol, dp_hol, dm_hol, OBJ_HOL,
               P_DA, P_RT, P_PN, I, T, S)

MODEL COMPARISON ANALYSIS

1. OBJECTIVE VALUE COMPARISON
   Individual Model:   1600896.79
   Endogenous Model:   1508674.61
   Holistic Model:     1730223.49

2. INTERNAL POOL MARKET CLEARING CHECK
   Daily Total:
     Total Sell Value:   5263724.71
     Total Buy Value:    5263724.71
     Net Transfer:             0.00
     Balanced: ✓

   Hourly Check:
     All hourly balances satisfied ✓

   Quantity Balance Check:
     All quantity balances satisfied ✓

3. DAY-AHEAD PROFIT COMPARISON
   Endogenous Model:    639580.85
   Holistic Model:      296729.91
   Difference:          342850.94

4. DAY-AHEAD COMMITMENT QUANTITY
   Endogenous Model:      4165.00
   Holistic Model:        1961.00
   Difference:            2204.00

5. REAL-TIME PROFIT COMPARISON
   Endogenous Model:    883923.74
   Holistic Model:     1445814.46
   Difference:         -561890.71

6. REAL-TIME QUANTITY COMPARISON
   Endogenous Model:      4211.73
   Holistic Model:        6404.87
   Difference:           -2193.1

In [ ]:
for i, t, s in product(range(I), range(T), range(S)):
    if yp_hol[i, t, s] != 0 and dm_hol[i, t, s] != 0:
        print(i, t, s,'yp', yp_hol[i, t, s], 'dm', dm_hol[i, t, s])
    
    if dp_hol[i, t, s] != 0 and ym_hol[i, t, s] != 0:
        print(i, t, s,'dp', dp_hol[i, t, s], 'ym', ym_hol[i, t, s])

In [ ]:
for i, t, s in product(range(I), range(T), range(S)):
    if yp[i, t, s] != 0 and dm[i, t, s] != 0:
        print(i, t, s,'yp', yp[i, t, s], 'dm', dm[i, t, s])
    
    if dp[i, t, s] != 0 and ym[i, t, s] != 0:
        print(i, t, s,'dp', dp[i, t, s], 'ym', ym[i, t, s])

### Individual Replay

In [ ]:
def individual_replay(R, K, K0, P_DA, P_RT, P_PN, RHOP, RHOM, I, T, S, M1):
    
    model = gp.Model("DER_Individual_Replay")
    model.setParam("Heuristics", 0.2)
    model.setParam("TimeLimit", 60*15)
    
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") 
    ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")  
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")  
    dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")  
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") 
    zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1")
    phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2")
    phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")
    phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5")
    phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")
    
    model.update()
    
    obj = (
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] +
            RHOP[i, t, s] * dp[i, t, s] - RHOM[i, t, s] * dm[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S))
    )
    
    model.setObjective(obj, GRB.MAXIMIZE)
    
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        model.addConstr(zd[i, t, s] <= z[i, t, s])
        model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
        model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + zc[i, t, s] - zd[i, t, s])
        
        model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi2[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi3[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi4[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi5[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi6[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi7[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi7[i, t, s]))

    for i, s in product(range(I), range(S)):
        model.addConstr(z[i, 0, s] == K0[i])

    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {model.objVal}")
    else:
        print("No optimal solution found.")
    
    # Extract solution
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    return (x_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, dp_sol, dm_sol, model.objVal)

In [ ]:
RHOP, RHOM = np.full((I,T,S), -1e10), np.full((I,T,S), 1e10)

for i, t, s in product(range(I), range(T), range(S)):
    if dp[i, t, s] > 0 and dm[i, t, s] == 0:
        RHOP[i, t, s] = rhop[t]
        RHOM[i, t, s] = 1e10
    elif dm[i, t, s] > 0 and dp[i, t, s] == 0:
        RHOP[i, t, s] = -1e10
        RHOM[i, t, s] = rhom[t] 

x_re, yp_re, ym_re, z_re, zc_re, zd_re, dp_re, dm_re, OBJ_RE = individual_replay(R, K, K0, P_DA, P_RT, P_PN, RHOP, RHOM, I, T, S, M1)

Set parameter Heuristics to value 0.2
Set parameter TimeLimit to value 900
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  900
Heuristics  0.2

Optimize a model with 72150 rows, 50670 columns and 172950 nonzeros
Model fingerprint: 0xcfeb6a07
Variable types: 25470 continuous, 25200 integer (25200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e+00, 3e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]
Found heuristic solution: objective -3.26783e+13
Presolve removed 63428 rows and 44052 columns
Presolve time: 2.46s
Presolved: 8722 rows, 6618 columns, 21755 nonzeros
Found heuristic solution: objective -7.77900e+12
Variable types: 3363 continuous, 3255 integer (3255 binary)

Root relaxation: objective 

In [ ]:
# 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    46.37     0.00    13.50     0.00     0.00     0.00    36.90     4.03    15.67
 9 |   169.57     7.00    18.47     0.00     6.00     0.00   140.60     2.50    48.53
10 |   438.10   258.00    65.63     0.00    10.37    40.57   178.37    33.70   186.63
11 |   661.03     0.00   580.53     0.00    16.27     0.00   127.40    63.17   331.30
12 |   875.37     0.00   731.23     0.00    89.03     0.00   102.87    47.77   395.53
13 |  1259.70     0.00  1484.00     0.00   116.00     0.00    23.50   363.80   450.63
14 |  1365.03     0.00   239.37     0.00   763.87     0.00   371.30     9.50   110.33
15 |   868.27     0.00  1299.40     0.00    -0.00     0.00     1.47   432.60   472.13
16 |   800.70   593.00    96.43     1.80    24.97   103.90   193.47     1.47    41.00
17 |   763.67     0.00   935.43     0.00    11.97

In [ ]:
# 한 명 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)
i=0
for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[i, t, s] for s in range(S)])
    x_sum = x_re[i, t]
    yp_avg = np.mean([yp_re[i, t, s] for s in range(S)])
    ym_avg = np.mean([ym_re[i, t, s] for s in range(S)])
    dp_avg = np.mean([dp_re[i, t, s] for s in range(S)])
    dm_avg = np.mean([dm_re[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_re[i, t, s] for s in range(S)])
    zd_avg = np.mean([zd_re[i, t, s] for s in range(S)])
    z_avg = np.mean([z_re[i, t, s] for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    19.97     0.00     0.00     0.00     0.00     0.00    19.97     0.00     0.00
 9 |     3.60     0.00     0.00     0.00     0.00     0.00     3.60     0.00    19.97
10 |    35.17     0.00     0.23     0.00     0.00     0.00    34.93     0.00    23.57
11 |    71.50     0.00    46.07     0.00     0.00     0.00    31.27     5.83    58.50
12 |   136.27     0.00    87.27     0.00    52.93     0.00    15.27    19.20    83.93
13 |   213.20     0.00   213.03     0.00    69.13     0.00     3.50    72.47    80.00
14 |   175.17     0.00    89.37     0.00     0.00     0.00    85.80     0.00    11.03
15 |   175.93     0.00   261.77     0.00     0.00     0.00     0.00    85.83    96.83
16 |    19.73     0.00     4.03     0.00     0.00     0.00    15.70     0.00    11.00
17 |     0.00     0.00    25.83     0.00     0.00